In [1]:
import os
import csv
import numpy as np
import pandas as pd
import gc



In [2]:
BASE_DIRS = [
    "/kaggle/input/h-and-m-personalized-fashion-recommendations",
    "/kaggle/input",
    "/kaggle/data/h-and-m-personalized-fashion-recommendations",
    "/kaggle/data",
]


def find_file(filename: str) -> str:
    for b in BASE_DIRS:
        p = os.path.join(b, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find {filename} in any of: {BASE_DIRS}")


transactions_path = find_file("transactions_train.csv")
sample_path = find_file("sample_submission.csv")

transactions_path, sample_path




('/kaggle/input/h-and-m-personalized-fashion-recommendations/transactions_train.csv',
 '/kaggle/input/h-and-m-personalized-fashion-recommendations/sample_submission.csv')

In [3]:
# Change (score-critical for 0.0->nonzero): make customer_id lookup keys robust and consistent
# across sample and transactions (strip + lowercase), while still outputting sample IDs verbatim.
def norm_cust_key(series: pd.Series) -> pd.Series:
    s = series.astype("object")
    s = s.where(pd.notna(s), None)
    s = s.map(lambda x: x.strip() if isinstance(x, str) else None)
    s = s.map(lambda x: x.lower() if isinstance(x, str) else None)
    s = s.map(lambda x: x if (x is not None and x != "") else None)
    return pd.Series(s, index=series.index, dtype="object")


# Change (stability): parse article_id deterministically as int from digits-only string.
def parse_article_id(series: pd.Series) -> pd.Series:
    s = series.astype("object").where(pd.notna(series), None)
    s = s.map(lambda x: x.strip() if isinstance(x, str) else None)
    s = s.map(
        lambda x: (
            "".join(ch for ch in x if ch.isdigit()) if isinstance(x, str) else None
        )
    )
    out = pd.to_numeric(s, errors="coerce").astype("Int64")
    return out


sample = pd.read_csv(
    sample_path,
    usecols=["customer_id"],
    dtype={"customer_id": "object"},
    keep_default_na=False,
)
sample_customer_id_verbatim = sample["customer_id"].astype("object")
sample["customer_id_key"] = norm_cust_key(sample_customer_id_verbatim)

usecols = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
dtypes = {
    "t_dat": "object",
    "customer_id": "object",
    "article_id": "object",
    "sales_channel_id": "int8",
    "price": "float32",
}
trans = pd.read_csv(
    transactions_path,
    usecols=usecols,
    dtype=dtypes,
    keep_default_na=False,
)

trans["t_dat"] = pd.to_datetime(trans["t_dat"], format="%Y-%m-%d", errors="coerce")
trans["customer_id_key"] = norm_cust_key(trans["customer_id"])
trans["article_id"] = parse_article_id(trans["article_id"])

trans = trans.loc[
    trans["customer_id_key"].notna()
    & trans["t_dat"].notna()
    & trans["article_id"].notna()
].copy()
trans["article_id"] = trans["article_id"].astype("int64")

sample.shape, trans.shape



((1371980, 2), (31521960, 6))

In [4]:
max_date = trans["t_dat"].max()
recent_days = 14
recent_start = max_date - pd.Timedelta(days=recent_days)

recent = trans.loc[trans["t_dat"] >= recent_start, ["article_id"]]

top_global = (
    recent["article_id"].value_counts().head(500).index.astype("int64").tolist()
)

len(top_global), top_global[:5], max_date, recent_start



(500,
 [751471001, 915526001, 751471043, 918292001, 706016001],
 Timestamp('2020-09-14 00:00:00'),
 Timestamp('2020-08-31 00:00:00'))

In [5]:
trans_sorted = trans.sort_values(["customer_id_key", "t_dat"], ascending=[True, False])

cust_hist = (
    trans_sorted.drop_duplicates(["customer_id_key", "article_id"], keep="first")
    .groupby("customer_id_key")["article_id"]
    .apply(lambda s: s.head(50).astype("int64").tolist())
)

del trans_sorted
gc.collect()

cust_hist.head()



customer_id_key
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657    [568601043, 841260003, 887593002, 890498002, 7...
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa    [826211002, 599580055, 811835004, 723529001, 5...
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318    [858883002, 851400006, 750424014, 870304002, 8...
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e                               [742079001, 732413001]
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a    [896152002, 730683050, 927530004, 791587015, 5...
Name: article_id, dtype: object

In [6]:
top_global_str = [f"{int(a):010d}" for a in top_global]


def make_pred(cid_key: str) -> str:
    items = cust_hist.get(cid_key, None)
    rec = []
    seen = set()

    if items is not None:
        for a in items:
            s = f"{int(a):010d}"
            if s not in seen:
                seen.add(s)
                rec.append(s)
            if len(rec) >= 12:
                return " ".join(rec[:12])

    for s in top_global_str:
        if s not in seen:
            seen.add(s)
            rec.append(s)
        if len(rec) >= 12:
            break

    if len(rec) < 12:
        for s in top_global_str:
            if len(rec) >= 12:
                break
            if s not in seen:
                seen.add(s)
                rec.append(s)

    return " ".join(rec[:12])


sample_ids_key = sample["customer_id_key"].tolist()
preds = [make_pred(cidk) for cidk in sample_ids_key]

submission = pd.DataFrame(
    {
        "customer_id": sample_customer_id_verbatim.values,  # must be verbatim for evaluation matching
        "prediction": np.array(preds, dtype=object),
    }
)

submission.head(), submission.shape



(                                         customer_id  \
 0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
 1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
 2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
 3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
 4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   
 
                                           prediction  
 0  0568601043 0841260003 0887593002 0890498002 07...  
 1  0826211002 0599580055 0811835004 0723529001 05...  
 2  0858883002 0851400006 0750424014 0870304002 08...  
 3  0742079001 0732413001 0751471001 0915526001 07...  
 4  0896152002 0730683050 0927530004 0791587015 05...  ,
 (1371980, 2))

In [7]:
assert submission["customer_id"].isna().sum() == 0
assert submission["prediction"].isna().sum() == 0
assert submission.shape[0] == sample.shape[0]
assert (submission["prediction"].astype(str).str.split().str.len() == 12).all()
assert submission["prediction"].astype(str).str.contains(r"^\d{10}( \d{10}){11}$").all()

# Change (score-critical): guarantee output IDs match sample EXACTLY, order and values.
assert np.array_equal(
    submission["customer_id"].astype("object").values,
    sample_customer_id_verbatim.values,
)

submission["prediction"] = submission["prediction"].astype(str).str.strip()

submission.to_csv(
    "submission.csv",
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    quoting=csv.QUOTE_MINIMAL,
)

chk = pd.read_csv(
    "submission.csv",
    dtype={"customer_id": "object", "prediction": "object"},
    keep_default_na=False,
)
assert list(chk.columns) == ["customer_id", "prediction"]
assert chk.shape[0] == sample.shape[0]
assert np.array_equal(
    chk["customer_id"].astype("object").values, sample_customer_id_verbatim.values
)
assert (chk["prediction"].astype(str).str.strip().str.split().str.len() == 12).all()
assert (
    chk["prediction"]
    .astype(str)
    .str.strip()
    .str.contains(r"^\d{10}( \d{10}){11}$")
    .all()
)

print("Wrote submission.csv with shape:", chk.shape)
print(chk.head())



/tmp/ipykernel_11/2681327205.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  assert submission["prediction"].astype(str).str.contains(r"^\d{10}( \d{10}){11}$").all()
/tmp/ipykernel_11/2681327205.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"^\d{10}( \d{10}){11}$")


Wrote submission.csv with shape: (1371980, 2)
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0568601043 0841260003 0887593002 0890498002 07...  
1  0826211002 0599580055 0811835004 0723529001 05...  
2  0858883002 0851400006 0750424014 0870304002 08...  
3  0742079001 0732413001 0751471001 0915526001 07...  
4  0896152002 0730683050 0927530004 0791587015 05...  


In [8]:
for name in [
    "trans",
    "recent",
    "cust_hist",
    "preds",
    "submission",
    "sample",
    "top_global",
    "top_global_str",
    "chk",
    "sample_ids_key",
    "sample_customer_id_verbatim",
]:
    if name in globals():
        del globals()[name]
gc.collect()

61